<h1 align="center"><b> Brain Tumor Segmentation using Attention U-Net with ML Baseline, NLP & Explainable AI</b></h1>

# This project focuses on automated brain tumor segmentation from MRI scans using Attention U-Net, supported by a Random Forest baseline model. It also incorporates NLP techniques to analyze radiology reports and uses explainable AI methods like SHAP and Grad-CAM for interpretability. The goal is to develop an accurate, efficient, and clinically reliable system for real-time tumor detection and analysis.
</p>

# 1: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import shap
import warnings
warnings.filterwarnings('ignore')

# 2: Generate Synthetic MRI + Tumor Masks + Radiologist Reports

In [ ]:
def generate_sample():
    img = np.random.rand(128,128)
    mask = (img > 0.7).astype(np.float32)
    # Simulate a radiologist report
    reports = [
        "Tumor in left frontal lobe, enhancing, irregular borders.",
        "Well‑defined enhancing mass in temporal region.",
        "Heterogeneous lesion with central necrosis.",
        "No abnormal enhancement.",
        "Cystic lesion with solid mural nodule."
    ]
    report = np.random.choice(reports)
    return img[...,None], mask[...,None], report

n = 1000
X_img = []
y_mask = []
reports = []
for _ in range(n):
    img, mask, rep = generate_sample()
    X_img.append(img)
    y_mask.append(mask)
    reports.append(rep)
X_img = np.array(X_img)
y_mask = np.array(y_mask)
print(f"Images: {X_img.shape}, Masks: {y_mask.shape}")

# 3: Feature Extraction for ML Baseline

In [ ]:
def extract_features(img, mask):
    # Handcrafted features from image and mask
    area = np.sum(mask)
    perimeter = np.sum(np.abs(np.diff(mask, axis=0))) + np.sum(np.abs(np.diff(mask, axis=1)))
    intensity_mean = np.mean(img[mask>0])
    intensity_std = np.std(img[mask>0])
    return [area, perimeter, intensity_mean, intensity_std]

X_ml = np.array([extract_features(X_img[i,:,:,0], y_mask[i,:,:,0]) for i in range(n)])
y_ml = (np.sum(y_mask, axis=(1,2,3)) > 100).astype(int)  # binary: tumor present if mask area > 100
print(f"ML features shape: {X_ml.shape}")

# 4: Train ML Baseline (Random Forest with GridSearch)

In [ ]:
X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(X_ml, y_ml, test_size=0.2, random_state=42)
param_grid = {'n_estimators': [50,100], 'max_depth': [5,10,None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
grid.fit(X_train_ml, y_train_ml)
print(f"Best RF params: {grid.best_params_}, accuracy: {grid.best_score_:.4f}")
y_pred_ml = grid.best_estimator_.predict(X_test_ml)
print(classification_report(y_test_ml, y_pred_ml))

# 5: NLP on Radiologist Reports

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=20)
tfidf = vectorizer.fit_transform(reports)
terms = vectorizer.get_feature_names_out()
scores = tfidf.toarray().mean(axis=0)
term_df = pd.DataFrame({'Term': terms, 'Score': scores}).sort_values('Score', ascending=False)
fig = px.bar(term_df.head(10), x='Score', y='Term', orientation='h', title='TF‑IDF of Clinical Report Terms')
fig.show()

# Word cloud
wordcloud = WordCloud(width=800, height=400).generate(' '.join(reports))
plt.imshow(wordcloud); plt.axis('off'); plt.show()

# 6: Build Attention U‑Net (Deep Learning)

In [ ]:
def attention_gate(g, x, filters):
    g1 = layers.Conv2D(filters,1)(g)
    x1 = layers.Conv2D(filters,1)(x)
    psi = layers.Activation('relu')(g1 + x1)
    psi = layers.Conv2D(1,1,activation='sigmoid')(psi)
    return layers.multiply([x, psi])

def unet_attention():
    inputs = layers.Input((128,128,1))
    c1 = layers.Conv2D(64,3,activation='relu',padding='same')(inputs)
    p1 = layers.MaxPool2D(2)(c1)
    c2 = layers.Conv2D(128,3,activation='relu',padding='same')(p1)
    p2 = layers.MaxPool2D(2)(c2)
    b = layers.Conv2D(256,3,activation='relu',padding='same')(p2)
    u1 = layers.UpSampling2D(2)(b)
    a1 = attention_gate(u1, c2, 128)
    c3 = layers.Conv2D(128,3,activation='relu',padding='same')(tf.concat([u1,a1], axis=-1))
    u2 = layers.UpSampling2D(2)(c3)
    a2 = attention_gate(u2, c1, 64)
    c4 = layers.Conv2D(64,3,activation='relu',padding='same')(tf.concat([u2,a2], axis=-1))
    outputs = layers.Conv2D(1,1,activation='sigmoid')(c4)
    return models.Model(inputs,outputs)
model = unet_attention()

# 7: Dice Loss & Epoch Tracking Callback

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    return 1 - dice_coef(y_true, y_pred)

class EpochLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"Epoch {epoch+1}: loss={logs['loss']:.4f}, dice={logs['dice_coef']:.4f}, val_dice={logs['val_dice_coef']:.4f}")

# 8: Compile & Train with Epoch Tracking

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=dice_loss, metrics=[dice_coef])
early_stop = EarlyStopping(monitor='val_dice_coef', patience=10, mode='max', restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
history = model.fit(X_img, y_mask, validation_split=0.2, epochs=50, batch_size=16,
                    callbacks=[early_stop, reduce_lr, EpochLogger()], verbose=0)

# 9: Plot Training History

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Dice Coefficient'))
fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Val Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['dice_coef'], name='Train Dice'), row=1, col=2)
fig.add_trace(go.Scatter(y=history.history['val_dice_coef'], name='Val Dice'), row=1, col=2)
fig.update_layout(title='Training History', height=500)
fig.show()

# 10: Grad‑CAM Interpretability for CNN

In [ ]:
def grad_cam(model, img, layer_name='conv2d_8'):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img)
        loss = predictions[:,:,:,0]
    grads = tape.gradient(loss, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_output), axis=-1)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

sample_img = X_img[0:1]
heatmap = grad_cam(model, sample_img, 'conv2d_8')
plt.imshow(sample_img[0,:,:,0], cmap='gray')
plt.imshow(heatmap[0], cmap='jet', alpha=0.5)
plt.title('Grad‑CAM Overlay'); plt.show()

# 11: SHAP for ML Baseline

In [ ]:
explainer = shap.TreeExplainer(grid.best_estimator_)
shap_values = explainer.shap_values(X_test_ml)
shap.summary_plot(shap_values, X_test_ml, feature_names=['Area', 'Perimeter', 'Intensity Mean', 'Intensity Std'], show=False)
plt.title('SHAP Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

# 12: Hyperparameter Tuning for DL (Learning Rate)

In [ ]:
# Simple grid for learning rate
lr_candidates = [1e-4, 1e-3, 1e-2]
val_dice = []
for lr in lr_candidates:
    model_tmp = unet_attention()
    model_tmp.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=dice_loss, metrics=[dice_coef])
    hist = model_tmp.fit(X_img[:200], y_mask[:200], validation_split=0.2, epochs=5, verbose=0)
    val_dice.append(hist.history['val_dice_coef'][-1])
best_lr = lr_candidates[np.argmax(val_dice)]
print(f"Best learning rate: {best_lr}")

# 13: Cross‑Validation for DL (using smaller subset)

In [ ]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=3, shuffle=True, random_state=42)
cv_scores = []
for train_idx, val_idx in kfold.split(X_img[:300]):
    model_cv = unet_attention()
    model_cv.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=dice_loss, metrics=[dice_coef])
    model_cv.fit(X_img[train_idx], y_mask[train_idx], epochs=10, verbose=0)
    _, dice = model_cv.evaluate(X_img[val_idx], y_mask[val_idx], verbose=0)
    cv_scores.append(dice)
print(f"3‑fold CV Dice: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

# 14: Save Best DL Model

In [ ]:
model.save('brain_tumor_attention_unet.h5')
print("Model saved.")

# 15: Runtime Prediction Cell (Upload MRI + Report)

In [ ]:
def predict_tumor(image_path, report_text=None):
    from tensorflow.keras.preprocessing import image
    img = image.load_img(image_path, target_size=(128,128), color_mode='grayscale')
    img = image.img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)
    pred_mask = model.predict(img)[0,:,:,0]
    plt.imshow(pred_mask, cmap='hot')
    plt.title('Predicted Tumor Mask')
    plt.show()

    if report_text:
        tfidf_report = vectorizer.transform([report_text])
        # You could also run a classifier on the report (e.g., sentiment/severity)
        print(f"Report TF‑IDF vector shape: {tfidf_report.shape}")
    return pred_mask

# Example usage (uncomment to test with a file)
# predict_tumor('mri_slice.png', 'Large enhancing tumor in frontal lobe')

print("Real Dataset: BraTS 2021 - https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1")